# AstroCLIMB graph resolver

This notebook uses the full `adsabs/AstroCLIMB` dataset as an entity and citation-graph index. It resolves Kaggle objects to Hugging Face figure records and reconstructs labels from `UUID`, `Paper DOI`, `References DOIs`, and `Citing DOIs`.

The workflow is deliberately staged:

1. Scan Kaggle train/test once and store compact object fingerprints in SQLite.
2. Stream Hugging Face text metadata without the 72 GB image column and resolve captions.
3. Evaluate deterministic graph predictions on the Kaggle training pairs.
4. Optionally stream images to resolve still-unmatched figures by canonical pixel hash.
5. Produce a hybrid test submission by overriding a fallback model only when the graph result is unambiguous.

All expensive stages are resumable. Recommended Kaggle settings: Internet enabled, persistent working files, high-RAM CPU runtime for metadata indexing, and sufficient disk space for the Kaggle CSVs. The optional image pass can transfer most of the 72.4 GB Hugging Face dataset.

In [ ]:
# Uncomment if needed.
# !pip install -q datasets huggingface_hub pillow

In [ ]:
import base64, hashlib, io, json, re, sqlite3, time, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from datasets import Image as HFImage
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

LABELS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
WORK = Path('/kaggle/working/astroclimb_graph')
WORK.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK / 'resolver.sqlite'
CSV_CHUNK_SIZE = 8
HF_DATASET = 'adsabs/AstroCLIMB'
RUN_IMAGE_PASS = False  # Set True after inspecting caption-only coverage.

# Optional prior submission from the improved model. Graph predictions override it.
# Set to a file under /kaggle/input or /kaggle/working. Leave None for a class-prior fallback.
FALLBACK_SUBMISSION = None

def find_file(filename):
    matches = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename} under /kaggle/input')
    print(filename, 'matches:', [str(p) for p in matches])
    return matches[0]

TRAIN_CSV = find_file('train.csv')
TEST_CSV = find_file('test.csv')
SAMPLE_CSV = find_file('sample_submission.csv')

## Canonical fingerprints

Captions use Unicode NFKC normalization, case folding, and collapsed whitespace. Images use a SHA-256 digest over dimensions plus canonical RGB pixels, so different PNG compression produces the same fingerprint. The raw Base64 value is never used as a database key.

In [ ]:
def is_png_b64(value):
    return isinstance(value, str) and value.lstrip().startswith('iVBOR')

def normalize_caption(value):
    text = unicodedata.normalize('NFKC', str(value or '')).casefold()
    return re.sub(r'\s+', ' ', text).strip()

def sha256_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def caption_hash(value):
    return sha256_text(normalize_caption(value))

def canonical_pixel_hash(image):
    image = image.convert('RGB')
    payload = image.width.to_bytes(4, 'big') + image.height.to_bytes(4, 'big') + image.tobytes()
    return hashlib.sha256(payload).hexdigest()

def kaggle_pixel_hash(value):
    raw = base64.b64decode(value.strip(), validate=False)
    with Image.open(io.BytesIO(raw)) as image:
        return canonical_pixel_hash(image)

def normalize_doi(value):
    text = str(value or '').strip().casefold()
    for prefix in ('https://doi.org/', 'http://doi.org/', 'http://dx.doi.org/', 'doi:'):
        if text.startswith(prefix):
            text = text[len(prefix):]
            break
    return text.rstrip('.,; ')

def normalize_doi_list(values):
    return sorted({doi for doi in (normalize_doi(x) for x in (values or [])) if doi})

## Persistent SQLite index

Only compact hashes, IDs, metadata, and matches are stored. `object_matches` is many-to-many because captions can be duplicated. Exact image hashes are usually unique, but the same representation safely handles duplicates.

In [ ]:
db = sqlite3.connect(DB_PATH)
db.execute('PRAGMA journal_mode=WAL')
db.execute('PRAGMA synchronous=NORMAL')
db.executescript('''
CREATE TABLE IF NOT EXISTS objects (
    object_hash TEXT PRIMARY KEY,
    modality TEXT NOT NULL,
    fingerprint TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_objects_fingerprint ON objects(modality, fingerprint);

CREATE TABLE IF NOT EXISTS records (
    uuid TEXT PRIMARY KEY,
    image_id TEXT,
    doi TEXT NOT NULL,
    references_json TEXT NOT NULL,
    citing_json TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_records_doi ON records(doi);

CREATE TABLE IF NOT EXISTS object_matches (
    object_hash TEXT NOT NULL,
    uuid TEXT NOT NULL,
    method TEXT NOT NULL,
    confidence REAL NOT NULL,
    PRIMARY KEY (object_hash, uuid, method)
);
CREATE INDEX IF NOT EXISTS idx_matches_object ON object_matches(object_hash);

CREATE TABLE IF NOT EXISTS progress (
    stage TEXT PRIMARY KEY,
    completed INTEGER NOT NULL,
    detail TEXT
);
''')
db.commit()

def stage_complete(stage):
    row = db.execute('SELECT completed FROM progress WHERE stage=?', (stage,)).fetchone()
    return bool(row and row[0])

def finish_stage(stage, detail=''):
    db.execute('INSERT OR REPLACE INTO progress(stage, completed, detail) VALUES (?, 1, ?)', (stage, detail))
    db.commit()

## Scan Kaggle objects

The CSV scan computes one fingerprint for every unique object. SHA-256 over the original value is used only as a compact internal object identifier; the full caption/Base64 payload is not retained. Repeated objects are fingerprinted once within each small chunk and deduplicated by SQLite across chunks.

In [ ]:
def internal_object_hash(value):
    value = value if isinstance(value, str) else ''
    prefix = 'I\0' if is_png_b64(value) else 'T\0'
    return sha256_text(prefix + value)

def scan_kaggle_objects(csv_path, stage):
    if stage_complete(stage):
        print(stage, 'already complete')
        return
    started = time.time()
    seen_rows = 0
    for part, chunk in enumerate(pd.read_csv(csv_path, usecols=['obj_1', 'obj_2'], chunksize=CSV_CHUNK_SIZE)):
        pending = {}
        for column in ('obj_1', 'obj_2'):
            for raw in chunk[column].fillna(''):
                value = raw if isinstance(raw, str) else ''
                key = internal_object_hash(value)
                pending.setdefault(key, value)
        existing = set()
        keys = list(pending)
        for start in range(0, len(keys), 900):
            batch = keys[start:start + 900]
            marks = ','.join('?' for _ in batch)
            existing.update(row[0] for row in db.execute(
                f'SELECT object_hash FROM objects WHERE object_hash IN ({marks})', batch))
        rows = []
        for key, value in pending.items():
            if key in existing:
                continue
            if is_png_b64(value):
                try:
                    fingerprint = kaggle_pixel_hash(value)
                except Exception as exc:
                    print('Image decode warning:', exc)
                    fingerprint = sha256_text('decode-failed:' + key)
                modality = 'image'
            else:
                modality, fingerprint = 'text', caption_hash(value)
            rows.append((key, modality, fingerprint))
        db.executemany('INSERT OR IGNORE INTO objects VALUES (?, ?, ?)', rows)
        db.commit()
        seen_rows += len(chunk)
        if part % 100 == 0:
            print(stage, 'rows', seen_rows, 'minutes', round((time.time()-started)/60, 1))
    finish_stage(stage, f'{seen_rows} pair rows scanned')

scan_kaggle_objects(TRAIN_CSV, 'scan_train')
scan_kaggle_objects(TEST_CSV, 'scan_test')
print(pd.read_sql_query('SELECT modality, COUNT(*) AS unique_objects FROM objects GROUP BY modality', db))

## Stream Hugging Face metadata and resolve captions

The image column is excluded from this pass. Each row is stored once as compact graph metadata. Only caption hashes that occur in Kaggle are inserted into the match table.

In [ ]:
def index_hf_metadata():
    if stage_complete('hf_metadata'):
        print('Hugging Face metadata already indexed')
        return
    target_caption_hashes = {row[0] for row in db.execute(
        "SELECT DISTINCT fingerprint FROM objects WHERE modality='text'")}
    stream = load_dataset(HF_DATASET, split='train', streaming=True)
    columns = ['UUID', 'Image ID', 'Paper DOI', 'Image Caption', 'References DOIs', 'Citing DOIs']
    stream = stream.select_columns(columns)
    record_rows, match_rows = [], []
    started = time.time()
    count = 0
    for row in stream:
        uuid = str(row['UUID'])
        doi = normalize_doi(row['Paper DOI'])
        record_rows.append((uuid, str(row.get('Image ID') or ''), doi,
                            json.dumps(normalize_doi_list(row.get('References DOIs'))),
                            json.dumps(normalize_doi_list(row.get('Citing DOIs')))))
        fingerprint = caption_hash(row.get('Image Caption') or '')
        if fingerprint in target_caption_hashes:
            object_rows = db.execute(
                "SELECT object_hash FROM objects WHERE modality='text' AND fingerprint=?", (fingerprint,))
            match_rows.extend((object_hash, uuid, 'caption_exact', 1.0) for (object_hash,) in object_rows)
        count += 1
        if len(record_rows) >= 1000:
            db.executemany('INSERT OR REPLACE INTO records VALUES (?, ?, ?, ?, ?)', record_rows)
            db.executemany('INSERT OR IGNORE INTO object_matches VALUES (?, ?, ?, ?)', match_rows)
            db.commit(); record_rows.clear(); match_rows.clear()
        if count % 10000 == 0:
            print('HF metadata rows', count, 'minutes', round((time.time()-started)/60, 1))
    db.executemany('INSERT OR REPLACE INTO records VALUES (?, ?, ?, ?, ?)', record_rows)
    db.executemany('INSERT OR IGNORE INTO object_matches VALUES (?, ?, ?, ?)', match_rows)
    db.commit()
    finish_stage('hf_metadata', f'{count} records indexed')

index_hf_metadata()

## Optional exact-image resolution

Run this only after examining caption-only coverage. Casting the image feature with `decode=False` exposes its bytes/path, then PIL performs canonical RGB decoding. This pass may transfer most of the 72.4 GB dataset. It is resumable at the stage level, but an interrupted stream restarts; for production, save a full image-hash index as a reusable private Kaggle Dataset.

In [ ]:
def hf_image_to_pil(value):
    if isinstance(value, Image.Image):
        return value
    if isinstance(value, dict):
        if value.get('bytes') is not None:
            return Image.open(io.BytesIO(value['bytes']))
        if value.get('path'):
            return Image.open(value['path'])
    raise TypeError(f'Unsupported Hugging Face image value: {type(value)}')

def index_hf_images():
    if stage_complete('hf_images'):
        print('Hugging Face images already indexed')
        return
    target_hashes = {row[0] for row in db.execute(
        "SELECT DISTINCT fingerprint FROM objects WHERE modality='image'")}
    stream = load_dataset(HF_DATASET, split='train', streaming=True)
    stream = stream.cast_column('image', HFImage(decode=False)).select_columns(['UUID', 'image'])
    started = time.time(); count = 0; matches = 0
    for row in stream:
        try:
            with hf_image_to_pil(row['image']) as image:
                fingerprint = canonical_pixel_hash(image)
        except Exception as exc:
            print('HF image warning at', row.get('UUID'), exc)
            continue
        if fingerprint in target_hashes:
            object_rows = db.execute(
                "SELECT object_hash FROM objects WHERE modality='image' AND fingerprint=?", (fingerprint,))
            rows = [(object_hash, str(row['UUID']), 'pixel_exact', 1.0) for (object_hash,) in object_rows]
            db.executemany('INSERT OR IGNORE INTO object_matches VALUES (?, ?, ?, ?)', rows)
            matches += len(rows)
        count += 1
        if count % 1000 == 0: db.commit()
        if count % 10000 == 0:
            print('HF image rows', count, 'matches', matches, 'minutes', round((time.time()-started)/60, 1))
    db.commit()
    finish_stage('hf_images', f'{count} images indexed; {matches} matches')

if RUN_IMAGE_PASS:
    index_hf_images()
else:
    print('Image pass skipped. Set RUN_IMAGE_PASS=True to enable it.')

## Coverage report

In [ ]:
coverage = pd.read_sql_query('''
SELECT o.modality,
       COUNT(*) AS objects,
       SUM(CASE WHEN n.matches > 0 THEN 1 ELSE 0 END) AS resolved,
       SUM(CASE WHEN n.matches = 1 THEN 1 ELSE 0 END) AS uniquely_resolved,
       AVG(CASE WHEN n.matches > 0 THEN 1.0 ELSE 0.0 END) AS coverage
FROM objects o
LEFT JOIN (SELECT object_hash, COUNT(DISTINCT uuid) AS matches
           FROM object_matches GROUP BY object_hash) n USING(object_hash)
GROUP BY o.modality
''', db)
display(coverage)

ambiguous = pd.read_sql_query('''
SELECT o.modality, m.object_hash, COUNT(DISTINCT m.uuid) AS candidates
FROM object_matches m JOIN objects o USING(object_hash)
GROUP BY m.object_hash HAVING candidates > 1
ORDER BY candidates DESC LIMIT 20
''', db)
display(ambiguous)

## Pair-level graph inference

For ambiguous object matches, every candidate UUID combination is evaluated. A graph label is accepted only when all candidate combinations agree. This conservative rule avoids overwriting a learned prediction with an uncertain metadata lookup.

In [ ]:
def get_candidates(object_hash):
    rows = db.execute('''
        SELECT r.uuid, r.doi, r.references_json, r.citing_json, m.confidence
        FROM object_matches m JOIN records r USING(uuid)
        WHERE m.object_hash=? ORDER BY m.confidence DESC
    ''', (object_hash,)).fetchall()
    return [{'uuid': uuid, 'doi': doi, 'references': set(json.loads(refs)),
             'citing': set(json.loads(citing)), 'confidence': confidence}
            for uuid, doi, refs, citing, confidence in rows]

def graph_label(left, right, left_modality, right_modality):
    if left['uuid'] == right['uuid'] and left_modality != right_modality:
        return 'same_figure'
    if left['doi'] and left['doi'] == right['doi']:
        return 'same_paper'
    linked = (right['doi'] in left['references'] or left['doi'] in right['references'] or
              right['doi'] in left['citing'] or left['doi'] in right['citing'])
    return 'related_papers' if linked else 'unrelated_papers'

def resolve_pair(value_a, value_b):
    hash_a, hash_b = internal_object_hash(value_a), internal_object_hash(value_b)
    modality_a = 'image' if is_png_b64(value_a) else 'text'
    modality_b = 'image' if is_png_b64(value_b) else 'text'
    candidates_a, candidates_b = get_candidates(hash_a), get_candidates(hash_b)
    if not candidates_a or not candidates_b:
        return None, 0.0, len(candidates_a), len(candidates_b)
    labels = {graph_label(a, b, modality_a, modality_b)
              for a in candidates_a for b in candidates_b}
    if len(labels) != 1:
        return None, 0.0, len(candidates_a), len(candidates_b)
    confidence = min(max(x['confidence'] for x in candidates_a),
                     max(x['confidence'] for x in candidates_b))
    return labels.pop(), confidence, len(candidates_a), len(candidates_b)

def graph_predictions(csv_path, has_labels):
    rows = []
    usecols = ['id', 'obj_1', 'obj_2'] + (LABELS if has_labels else [])
    for part, chunk in enumerate(pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE)):
        for record in chunk.itertuples(index=False):
            prediction, confidence, n_a, n_b = resolve_pair(record.obj_1, record.obj_2)
            item = {'id': record.id, 'graph_prediction': prediction,
                    'graph_confidence': confidence, 'candidates_a': n_a, 'candidates_b': n_b}
            if has_labels:
                item['target'] = LABELS[int(np.argmax([getattr(record, label) for label in LABELS]))]
            rows.append(item)
        if part % 100 == 0: print('pair chunks', part)
    return pd.DataFrame(rows)

## Evaluate on Kaggle training pairs

The resolved subset is the trustworthy test of whether the graph reconstruction and DOI normalization are correct. Coverage and resolved accuracy must be reported together; high resolved accuracy with low coverage is not yet a complete solution.

In [ ]:
train_graph_path = WORK / 'train_graph_predictions.csv'
if train_graph_path.exists():
    train_graph = pd.read_csv(train_graph_path)
else:
    train_graph = graph_predictions(TRAIN_CSV, has_labels=True)
    train_graph.to_csv(train_graph_path, index=False)

resolved = train_graph['graph_prediction'].notna()
print('Resolved pair coverage:', resolved.mean())
if resolved.any():
    truth = train_graph.loc[resolved, 'target']
    pred = train_graph.loc[resolved, 'graph_prediction']
    print('Resolved accuracy:', accuracy_score(truth, pred))
    print('Resolved macro-F1:', f1_score(truth, pred, labels=LABELS, average='macro'))
    print(classification_report(truth, pred, labels=LABELS, target_names=LABELS, digits=4, zero_division=0))
    display(pd.DataFrame(confusion_matrix(truth, pred, labels=LABELS), index=LABELS, columns=LABELS))

display(pd.crosstab(train_graph['target'], train_graph['graph_prediction'], margins=True, dropna=False))

## Create a hybrid submission

Attach the improved model's submission and set `FALLBACK_SUBMISSION` for the recommended hybrid. Exact, unambiguous graph predictions override it. If no fallback is supplied, unresolved pairs receive the most common Kaggle training class; that mode is intended only to validate the submission pipeline, not as a competitive final entry.

In [ ]:
test_graph_path = WORK / 'test_graph_predictions.csv'
if test_graph_path.exists():
    test_graph = pd.read_csv(test_graph_path)
else:
    test_graph = graph_predictions(TEST_CSV, has_labels=False)
    test_graph.to_csv(test_graph_path, index=False)

sample = pd.read_csv(SAMPLE_CSV)
if FALLBACK_SUBMISSION is not None:
    submission = pd.read_csv(FALLBACK_SUBMISSION)
    submission = sample[['id']].merge(submission, on='id', how='left')
else:
    counts = pd.read_csv(TRAIN_CSV, usecols=LABELS)[LABELS].sum()
    majority = counts.idxmax()
    submission = sample[['id']].copy()
    for label in LABELS:
        submission[label] = np.int8(label == majority)
    print('Warning: using majority fallback:', majority)

graph_by_id = test_graph.set_index('id')['graph_prediction']
overrides = 0
for row_index, sample_id in enumerate(submission['id']):
    label = graph_by_id.get(sample_id)
    if isinstance(label, str) and label in LABELS:
        submission.loc[row_index, LABELS] = 0
        submission.loc[row_index, label] = 1
        overrides += 1

submission[LABELS] = submission[LABELS].astype(np.int8)
assert submission[LABELS].notna().all().all()
assert (submission[LABELS].sum(axis=1) == 1).all()
output_path = '/kaggle/working/submission_graph_hybrid.csv'
submission.to_csv(output_path, index=False)
print('Graph overrides:', overrides, '/', len(submission), '=', overrides / len(submission))
print('Saved:', output_path)
display(submission.head())

## Interpretation and next steps

- If resolved accuracy is high but image coverage is low, enable the exact-image pass.
- If exact-image matching still misses objects, add a separate approximate resolver using pHash/DINOv2 retrieval and accept matches only above a threshold calibrated on Kaggle training objects.
- If captions are ambiguous, use the other object in the pair and graph consistency to rerank candidates rather than selecting a caption candidate arbitrarily.
- Preserve a fallback learned model for pairs where either object is unresolved or candidate graph labels disagree.
- Do not claim the resolved-subset score as overall validation performance; always report coverage alongside it.
- After a successful run, save `resolver.sqlite` and prediction CSVs as a private Kaggle Dataset so future notebooks avoid repeating the Hugging Face scans.